In [6]:
#installation the packges
!pip install -U langchain langchain-groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.7.0
    Uninstalling groq-1.7.0:
      Successfully uninstalled groq-1.7.0


In [2]:
# !pip install openai

In [ ]:
# First check how many models are there

import os
from groq import Groq

# Initialize Groq API key
os.environ["GROQ_API_KEY"] = ""

client = Groq()

models = client.models.list()

for model in models.data:
    print(model.id)

canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
qwen/qwen3.8-27b
openai/gpt-oss-20b
groq/compound-mini
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
qwen/qwen3.6-27b
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
groq/compound
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m


In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.6
)

response = llm.invoke("What is the capital of India?")

print(response.content)

The capital of India is **New Delhi**.


In [8]:
#Zero shot prompting
#without giving any point through example
response1 = llm.invoke(
    "Classify this review as positive, negative, or neutral: "
    "The laptop battery is excellent."
)

print(response1.content)

Positive.


In [9]:
#role based Prompting
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(
        content="You are an expert PostgreSQL instructor."
    ),
    HumanMessage(
        content="Explain INNER JOIN with an example."
    )
]

response = llm.invoke(messages)

print(response.content)

## What is an **INNER JOIN**?

In PostgreSQL (and in SQL in general), an **INNER JOIN** returns only the rows that have matching values in **both** tables you are joining.  
Think of it as the intersection of two sets: you keep the rows where the join condition is true, and discard everything else.

### Visual metaphor

```
Table A          Table B
+----+------+    +----+------+
| id | name |    | id | city |
+----+------+    +----+------+
| 1  | Alice|    | 1  | NY   |
| 2  | Bob  |    | 2  | LA   |
| 3  | Carol|    | 4  | SF   |
+----+------+    +----+------+

Result of A INNER JOIN B on A.id = B.id
+----+------+------+
| id | name | city |
+----+------+------+
| 1  | Alice| NY   |
| 2  | Bob  | LA   |
+----+------+------+
```

Rows with `id = 3` (Carol) and `id = 4` (SF) disappear because they have no counterpart in the other table.

---

## Syntax

```sql
SELECT <columns>
FROM   table1
INNER JOIN table2
        ON table1.column = table2.column;
```

- `INNER` is optional – `JOIN` a

In [10]:
#few shot prompting where we are giving a set of example to LLM and then we will ask them to answer for certain question
prompt = """
Classify the sentiment.

Example 1:
Review: "The product is amazing."
Sentiment: Positive

Example 2:
Review: "The product is terrible."
Sentiment: Negative

Example 3:
Review: "The product is okay."
Sentiment: Neutral

Now classify:

Review: "The product exceeded my expectations."
Sentiment:
"""

response2 = llm.invoke(prompt)

print(response2.content)

Positive


In [11]:
#prompt template which we can make once and we can use how many times we want
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert {subject} instructor."
    ),
    (
        "human",
        "Explain {topic} to a {level} student."
    )
])


chain = prompt | llm

response3 = chain.invoke({
    "subject": "Machine Learning",
    "topic": "Random Forest",
    "level": "beginner"
})

print(response3.content)

## Random Forest – A Beginner‑Friendly Overview  

Imagine you have a big question to answer, like “Will this email be spam or not?”  
One way to answer it is to **ask a single expert** (a decision tree).  
Another way is to **ask a whole group of experts** (many decision trees) and let them vote.  
A *Random Forest* is exactly that: a **collection (forest) of many decision trees** that work together to make a more reliable prediction.

Below we’ll break down the idea step‑by‑step, using simple language, visual analogies, and a tiny code example at the end.

---

## 1. The Building Block – Decision Trees  

### What is a decision tree?  
- Think of a flowchart: at each node you ask a **yes/no (or “greater/less”) question** about a feature (e.g., “Is the email’s subject length > 30 characters?”).  
- Depending on the answer you move left or right, ask another question, and keep going until you reach a **leaf** that gives a final prediction (spam / not‑spam).  

### Why are trees easy to

In [12]:
#we have seen many more inputs now we will discuss about Structured Output
#typically we are using two structured output methods 1 Pydantic 2 Typing

from pydantic import BaseModel

class Person(BaseModel):
  name: str
  age: int
  occupation: str

structured_llm = llm.with_structured_output(Person)
response4 = structured_llm.invoke("Rahul is 25 year old and he is a Data Scientist")
print(response4)


name='Rahul' age=25 occupation='Data Scientist'


In [13]:
from langchain_groq import ChatGroq

llm2 = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

prompt = """
Solve the following problem using these steps:

Step 1: Find the number of AI employees.

Step 2: Find the number of Generative AI employees.

Step 3: Provide the final answer.

Problem:

A company has 120 employees.

25% work in AI.

40% of the AI employees work on Generative AI.

Return only the steps and final answer.
"""

response = llm2.invoke(prompt)

print(response.content)

Step 1: 120 employees × 25% = 30 AI employees.  
Step 2: 30 AI employees × 40% = 12 Generative AI employees.  
Step 3: Final answer: 12


In [14]:
# Prompt Versioning

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


prompts = {
    "v1": """
You are a helpful AI assistant.

Summarize the following text in 3 sentences.

Text:
{text}
""",

    "v2": """
You are an expert summarization assistant.

Summarize the following text.

Follow these rules:
- Identify the main idea.
- Include only important information.
- Remove unnecessary details.
- Use simple and clear language.
- Keep the summary below 80 words.

Text:
{text}
"""
}


# Function for Prompt Loading
def prompt_loader(version):
    if version not in prompts:
        raise ValueError(f"Prompt version {version} does not exist.")

    prompt = ChatPromptTemplate.from_template(
        prompts[version]
    )

    return prompt


# LLM creation
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


# Select prompt version
prompt_version = "v2"

prompt = prompt_loader(prompt_version)

In [15]:
chain = prompt | llm

text = """
Artificial intelligence is transforming many industries.
Companies are using AI for automation, healthcare,
fraud detection, recommendation systems and customer
support.

Generative AI has also made it possible to build
applications that understand natural language, generate
content, write code and interact with external tools.
"""


response = chain.invoke({
    "text": text
})

print(response.content)

Artificial intelligence is reshaping many sectors. Companies use AI for automation, healthcare, fraud detection, recommendation systems, and customer support. Generative AI lets apps understand language, create content, write code, and work with external tools.
